# Chapter 8: Data Visualisation
**Companion notebook** · Book 1: Python and Data Foundations for ML

Every code example from Chapter 8, in the same order as the book. Run the set-up cell first (needs an internet connection). Each chart appears below its cell.

## Set-up: load and prepare the data

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank_data = fetch_ucirepo(id=222)
bank = bank_data.data.features.copy()
bank["subscribed"] = bank_data.data.targets["y"]
bank = bank.fillna("unknown")

bank["subscribed_num"] = bank["subscribed"].map({"yes": 1, "no": 0})
bank["age_group"] = pd.cut(bank["age"], bins=[17, 29, 39, 49, 59, 100],
                           labels=["18-29", "30-39", "40-49", "50-59", "60+"])
print(bank.shape)

## 1. Matplotlib basics: a line chart

In [ ]:
import matplotlib.pyplot as plt

by_age = bank.groupby("age")["subscribed_num"].agg(["mean", "size"])
by_age = by_age[by_age["size"] >= 100]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_age.index, by_age["mean"] * 100)
ax.set_title("Subscription rate by age")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Subscription rate (%)")
plt.show()

## 2. Bar charts: comparing categories

In [ ]:
job_rates = bank.groupby("job")["subscribed_num"].mean().sort_values() * 100

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(job_rates.index, job_rates.values)
ax.set_title("Students and retired people subscribe most often")
ax.set_xlabel("Subscription rate (%)")
plt.show()

## 3. Histograms: the shape of one column

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bank["age"], bins=30, edgecolor="white")
ax.set_title("Most customers are aged 30 to 50")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of customers")
plt.show()

In [ ]:
shown = bank[bank["balance"].between(-2000, 10000)]
mean_balance = bank["balance"].mean()
median_balance = bank["balance"].median()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(shown["balance"], bins=60, edgecolor="white")
ax.axvline(median_balance, color="black", linestyle="--", label=f"Median: {median_balance:.0f}")
ax.axvline(mean_balance, color="red", label=f"Mean: {mean_balance:.0f}")
ax.set_title("Balances are heavily skewed: the mean sits far above the median")
ax.set_xlabel("Balance (euros), shown from -2,000 to 10,000")
ax.set_ylabel("Number of customers")
ax.legend()
plt.show()

print(f"Customers outside the range shown: {len(bank) - len(shown)}")

## 4. Seaborn: statistical charts in one line
### A box plot

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=bank, x="subscribed", y="age", ax=ax)
ax.set_title("Subscribers are spread more widely in age")
ax.set_xlabel("Subscribed to a term deposit")
ax.set_ylabel("Age (years)")
plt.show()

print(bank.groupby("subscribed")["age"].describe()[["25%", "50%", "75%"]])

### A scatter plot

In [ ]:
sample = bank.sample(2000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(data=sample, x="age", y="balance", hue="subscribed", alpha=0.5, ax=ax)
ax.set_title("Age and balance: no strong relationship")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Balance (euros)")
plt.show()

### Grouped bars with hue

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=bank, x="age_group", y="subscribed_num", hue="loan", errorbar=None, ax=ax)
ax.set_title("In every age group, customers without a personal loan subscribe more")
ax.set_xlabel("Age group")
ax.set_ylabel("Subscription rate")
ax.legend(title="Personal loan")
plt.show()

### A correlation heatmap

In [ ]:
numeric = bank[["age", "balance", "duration", "campaign", "pdays", "previous", "subscribed_num"]]
correlations = numeric.corr().round(2) + 0.0

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(correlations, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("How the numeric columns move together")
plt.show()

print(correlations["subscribed_num"].sort_values(ascending=False))

## 5. Telling a story with one chart

In [ ]:
rates = bank.groupby("age_group", observed=True)["subscribed_num"].mean() * 100
overall = bank["subscribed_num"].mean() * 100
colours = ["#d62728" if group == "60+" else "#9e9e9e" for group in rates.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(rates.index.astype(str), rates.values, color=colours)
ax.axhline(overall, color="black", linestyle="--", label=f"All customers: {overall:.1f}%")
ax.bar_label(bars, fmt="%.1f%%", label_type="center", color="white", fontweight="bold")
ax.legend(frameon=False)
ax.grid(False)
ax.set_title("Over-60s subscribe at almost three times the average rate")
ax.set_xlabel("Age group")
ax.set_ylabel("Subscription rate (%)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 6. Mini project: a one-page dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(bank["age"], bins=30, edgecolor="white")
axes[0, 0].set_title("Age of customers")
axes[0, 0].set_xlabel("Age (years)")

job_counts = bank["job"].value_counts().sort_values()
axes[0, 1].barh(job_counts.index, job_counts.values)
axes[0, 1].set_title("Customers by job")

housing_rates = bank.groupby("housing")["subscribed_num"].mean() * 100
axes[1, 0].bar(housing_rates.index, housing_rates.values, color=["#2ca02c", "#9e9e9e"])
axes[1, 0].set_title("Subscription rate (%) by housing loan")
axes[1, 0].set_xlabel("Has a housing loan")

calls = bank[bank["campaign"] <= 10].groupby("campaign")["subscribed_num"].mean() * 100
axes[1, 1].plot(calls.index, calls.values, marker="o")
axes[1, 1].set_title("Subscription rate (%) by number of calls")
axes[1, 1].set_xlabel("Calls made to the customer in this campaign")

fig.suptitle("Bank Marketing campaign: customer overview", fontsize=16)
fig.tight_layout()
fig.savefig("campaign_dashboard.png", dpi=150)
plt.show()

In [ ]:
print(housing_rates.round(1))
print(calls.round(1))

## Try it yourself
Use the empty cells below for the exercises at the end of Chapter 8.

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.